In [1]:
import os
import datetime
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)

import numpy as np
from laparoscopy_helpers.data_cleaning import (
    to_snake_case, clean_surgical_df
)
from laparoscopy_helpers.plots import *

In [2]:
os.listdir("../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger")

['Endoscopy Ledger_ 2022-2023.xlsx',
 'Endoscopy Ledger_Consolidated 2024.xlsx',
 'Endoscopy Ledger_2025 02.04.2026.xlsx']

In [3]:
cleaned_theatre_book = pd.read_excel("../Nkhoma_data/theatre_book_data/combined_clean.xlsx", engine="openpyxl")

In [4]:
endo_path = "../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger"
endo_files = os.listdir(endo_path)

In [5]:
endo_files

['Endoscopy Ledger_ 2022-2023.xlsx',
 'Endoscopy Ledger_Consolidated 2024.xlsx',
 'Endoscopy Ledger_2025 02.04.2026.xlsx']

In [6]:
for endo_file in endo_files:
    xl = pd.ExcelFile(f"{endo_path}/{endo_file}")
    print(f"{endo_file}: {xl.sheet_names}")

Endoscopy Ledger_ 2022-2023.xlsx: ['Endoscopy Camp 2022', '22 May', '22 June', '22 Aug', '22 Sept', '22 Oct', '22 Nov', '22 Dec', '23 JAN', '23 FEB', '23 MAR', '23 APR', '23 MAY', 'Endoscopy camp 2023', '23 SEPT', '23 OCT', '23 NOV', '23 DEC', 'ALL', 'FEB 2024']
Endoscopy Ledger_Consolidated 2024.xlsx: ['2024 FULL LIST', '2025', '2026']
Endoscopy Ledger_2025 02.04.2026.xlsx: ['January', 'February ', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']


In [7]:
sheets_to_load = {
    'Endoscopy Ledger_ 2022-2023.xlsx': [
        '22 May', '22 June', '22 Aug', '22 Sept', '22 Oct', '22 Nov', '22 Dec',
        '23 JAN', '23 FEB', '23 MAR', '23 APR', '23 MAY',
        '23 SEPT', '23 OCT', '23 NOV', '23 DEC'
        # skipping: 'Endoscopy Camp 2022', 'Endoscopy camp 2023', 'ALL', 'FEB 2024'
    ],
    'Endoscopy Ledger_Consolidated 2024.xlsx': ['2024 FULL LIST'],
    'Endoscopy Ledger_2025 02.04.2026.xlsx': [
        'January', 'February ', 'March', 'April', 'May', 'June',
        'July', 'August', 'September', 'October', 'November', 'December'
    ],
}
for filename, sheets in sheets_to_load.items():
    for sheet in sheets[:2]:  # just first 2 sheets per file to start
        df = pd.read_excel(f"{endo_path}/{filename}", sheet_name=sheet, header=None)
        print(f"\n{'='*60}")
        print(f"{filename} | {sheet}")
        print(df.head(10).to_string())


Endoscopy Ledger_ 2022-2023.xlsx | 22 May
                             0                    1                    2                3            4             5
0  ENDOSCOPY PATIENTS MAY 2022                  NaN                  NaN              NaN          NaN           NaN
1                          NaN                  NaN                  NaN              NaN          NaN           NaN
2                         NAME    DATE OF TREATMENT    DATE OF DISCHARGE  TOTAL BILL (MK)  CO-PAY (MK)  BALANCE (MK)
3            Pitilizani Gidala  2022-05-11 00:00:00  2022-05-11 00:00:00            85250        22000         63250
4                Ellena Mulazi  2022-05-11 00:00:00  2022-05-12 00:00:00            83970            0         83970
5            Christina Manyowa  2022-05-11 00:00:00  2022-05-13 00:00:00            99770        65000         34770
6               Alice Jeremani  2022-05-18 00:00:00  2022-05-18 00:00:00            75820            0         75820
7             Catheri

In [8]:
import pandas as pd
import re

def to_snake_case(name):
    name = str(name).strip()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.lower()

def load_endo_sheet(filepath, sheet_name):
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None, engine="openpyxl")

    # Find header row: first row where first non-null value is '#' or 'NAME'
    header_idx = None
    for i, row in raw.iterrows():
        vals = row.dropna().tolist()
        if vals and str(vals[0]).strip() in ("#", "NAME"):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"No header row found in {sheet_name}")

    df = raw.iloc[header_idx + 1:].copy()
    df.columns = raw.iloc[header_idx].tolist()

    # Snake-case columns, handle '#' → 'id'
    df.columns = ["id" if str(c).strip() == "#" else to_snake_case(c) for c in df.columns]

    # Drop rows without a valid numeric id or name
    id_col = "id" if "id" in df.columns else "name"
    df = df[df[id_col].notna()]
    df = df[df[id_col].astype(str).str.strip() != ""]

    # Drop pure summary rows (all-NaN except one cell)
    df = df.dropna(how="all")

    df = df.reset_index(drop=True)
    return df


# ── Load everything ───────────────────────────────────────────────────
sheets_to_load = {
    'Endoscopy Ledger_ 2022-2023.xlsx': [
        '22 May', '22 June', '22 Aug', '22 Sept', '22 Oct', '22 Nov', '22 Dec',
        '23 JAN', '23 FEB', '23 MAR', '23 APR', '23 MAY',
        '23 SEPT', '23 OCT', '23 NOV', '23 DEC'
    ],
    'Endoscopy Ledger_Consolidated 2024.xlsx': ['2024 FULL LIST'],
    'Endoscopy Ledger_2025 02.04.2026.xlsx': [
        'January', 'February ', 'March', 'April', 'May', 'June',
        'July', 'August', 'September', 'October', 'November', 'December'
    ],
}

endos = []
for filename, sheets in sheets_to_load.items():
    for sheet in sheets:
        try:
            df = load_endo_sheet(f"{endo_path}/{filename}", sheet_name=sheet)
            df["source_file"] = filename
            df["source_sheet"] = sheet
            endos.append(df)
            print(f"✓ {sheet:20s} → {len(df)} rows | cols: {list(df.columns)}")
        except Exception as e:
            print(f"✗ {sheet:20s} → {e}")

✓ 22 May               → 14 rows | cols: ['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'balance_mk', 'source_file', 'source_sheet']
✓ 22 June              → 7 rows | cols: ['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'balance_mk', 'source_file', 'source_sheet']
✓ 22 Aug               → 14 rows | cols: ['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'balance_mk', 'source_file', 'source_sheet']
✓ 22 Sept              → 6 rows | cols: ['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'balance_mk', 'source_file', 'source_sheet']
✓ 22 Oct               → 5 rows | cols: ['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'balance_mk', 'source_file', 'source_sheet']
✓ 22 Nov               → 7 rows | cols: ['name', 'date_of_treatment', 'total_bill_mk', 'copay_mk', 'balance_mk', 'endoscopy_ledger', 'source_file', 'source_sheet']
✓ 22 Dec 

In [9]:
def normalize_endo_df(df):
    # Drop any column named 'nan'
    df = df.drop(columns=[c for c in df.columns if c == "nan"], errors="ignore")

    # Drop Mozambique-specific columns
    df = df.drop(columns=[c for c in df.columns if "moz" in c], errors="ignore")

    # Unify the balance column name → 'endoscopy_ledger_balance_mk'
    balance_aliases = {"balance_mk", "endoscopy_ledger", "bill_to_patientmk", "bill_to_patient"}
    for col in df.columns:
        if col in balance_aliases:
            df = df.rename(columns={col: "endoscopy_ledger_balance_mk"})

    # Deduplicate columns — keep first occurrence
    df = df.loc[:, ~df.columns.duplicated()]

    return df

# Apply and concat
endos_clean = [normalize_endo_df(df) for df in endos]
combined = pd.concat(endos_clean, ignore_index=True)

print(combined.shape)
print(combined.columns.tolist())

(553, 16)
['name', 'date_of_treatment', 'date_of_discharge', 'total_bill_mk', 'copay_mk', 'endoscopy_ledger_balance_mk', 'source_file', 'source_sheet', 'id', 'patient_number', 'gender', 'diagnosis', 'surgery', 'admission_date', 'discharge_date', 'invoice_number']


In [10]:
# Fill admission_date from date_of_treatment where missing
combined["admission_date"] = combined["admission_date"].fillna(combined["date_of_treatment"])
combined["discharge_date"] = combined["discharge_date"].fillna(combined["date_of_discharge"])

# Drop the old columns
combined = combined.drop(columns=["date_of_treatment", "date_of_discharge"])

In [11]:
cols = [
    "id", "patient_number", "name", "gender", "diagnosis", "surgery",
    "admission_date", "discharge_date", "total_bill_mk", "copay_mk",
    "endoscopy_ledger_balance_mk", "invoice_number", "source_file", "source_sheet"
]
combined = combined[cols]
print(combined.shape)
combined.head()

(553, 14)


,id,patient_number,name,gender,diagnosis,surgery,admission_date,discharge_date,total_bill_mk,copay_mk,endoscopy_ledger_balance_mk,invoice_number,source_file,source_sheet
0,NaN,NaN,Pitilizani Gidala,NaN,NaN,NaN,2022-05-11 00:00:00,2022-05-11 00:00:00,85250,22000,63250,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May
1,NaN,NaN,Ellena Mulazi,NaN,NaN,NaN,2022-05-11 00:00:00,2022-05-12 00:00:00,83970,0,83970,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May
2,NaN,NaN,Christina Manyowa,NaN,NaN,NaN,2022-05-11 00:00:00,2022-05-13 00:00:00,99770,65000,34770,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May
3,NaN,NaN,Alice Jeremani,NaN,NaN,NaN,2022-05-18 00:00:00,2022-05-18 00:00:00,75820,0,75820,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May
4,NaN,NaN,Catherine Samuel,NaN,NaN,NaN,2022-05-18 00:00:00,2022-05-19 00:00:00,100570,20570,80000,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May


In [12]:
combined["admission_date"] = pd.to_datetime(combined["admission_date"], errors="coerce")
combined["discharge_date"] = pd.to_datetime(combined["discharge_date"], errors="coerce")

# Now this works
print(combined["admission_date"].min(), "→", combined["admission_date"].max())

2022-04-07 00:00:00 → 2025-12-28 00:00:00


In [13]:
combined = combined[combined["name"].notna()].reset_index(drop=True)
print(combined.shape)

(491, 14)


In [14]:
print(combined.isnull().mean().round(2).sort_values(ascending=False))
print(combined["admission_date"].min(), "→", combined["admission_date"].max())

diagnosis                      0.34
id                             0.32
patient_number                 0.32
gender                         0.32
surgery                        0.32
invoice_number                 0.32
discharge_date                 0.06
admission_date                 0.05
copay_mk                       0.04
total_bill_mk                  0.04
endoscopy_ledger_balance_mk    0.02
name                           0.00
source_file                    0.00
source_sheet                   0.00
dtype: float64
2022-04-07 00:00:00 → 2025-12-28 00:00:00


In [15]:
# Are the remaining date/billing nulls all from the same sheets?
print(combined[combined["admission_date"].isna()][["source_sheet", "name", "total_bill_mk"]].to_string())

       source_sheet                                                                      name total_bill_mk
12           22 May                                                                     TOTAL           NaN
13           22 May                                              Approved by Dr. Jens Vaylann           NaN
19          22 June                                                                     TOTAL           NaN
20          22 June                                              Approved by Dr. Jens Vaylann           NaN
33           22 Aug                                                                     TOTAL           NaN
34           22 Aug                                              Approved by Dr. Jens Vaylann           NaN
39          22 Sept                                                                     TOTAL           NaN
40          22 Sept                                              Approved by Dr. Jens Vaylann           NaN
44           22 Oct         

In [16]:
combined = combined[combined["admission_date"].notna()].reset_index(drop=True)
print(combined.shape)
print(combined.isnull().mean().round(2).sort_values(ascending=False))

(468, 14)
diagnosis                      0.31
id                             0.29
patient_number                 0.29
gender                         0.29
surgery                        0.29
invoice_number                 0.29
discharge_date                 0.02
copay_mk                       0.01
name                           0.00
admission_date                 0.00
total_bill_mk                  0.00
endoscopy_ledger_balance_mk    0.00
source_file                    0.00
source_sheet                   0.00
dtype: float64


In [18]:
# Null rates per column
print(combined.isnull().mean().round(2).sort_values(ascending=False))

# Date range sanity
print(combined["admission_date"].min(), "→", combined["admission_date"].max())

# Duplicate names + dates (possible duplicate entries)
dupes = combined.duplicated(subset=["name", "admission_date"], keep=False)
print(f"\nPotential duplicates: {dupes.sum()}")
combined[dupes].sort_values("name")

diagnosis                      0.42
patient_number                 0.40
surgery                        0.40
gender                         0.40
invoice_number                 0.40
id                             0.29
discharge_date                 0.17
copay_mk                       0.15
total_bill_mk                  0.15
admission_date                 0.15
name                           0.11
endoscopy_ledger_balance_mk    0.05
source_file                    0.00
source_sheet                   0.00
dtype: float64
2022-04-07 00:00:00 → 2025-12-28 00:00:00

Potential duplicates: 114


,id,patient_number,name,gender,diagnosis,surgery,admission_date,discharge_date,total_bill_mk,copay_mk,endoscopy_ledger_balance_mk,invoice_number,source_file,source_sheet
78,NaN,NaN,Approved by Dr. Jens Vaylann,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_ 2022-2023.xlsx,23 FEB
13,NaN,NaN,Approved by Dr. Jens Vaylann,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 May
20,NaN,NaN,Approved by Dr. Jens Vaylann,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 June
103,NaN,NaN,Approved by Dr. Jens Vaylann,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_ 2022-2023.xlsx,23 APR
34,NaN,NaN,Approved by Dr. Jens Vaylann,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_ 2022-2023.xlsx,22 Aug
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,24,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
549,25,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
550,26,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
551,27,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December


In [19]:
# Who are the rows without a name?
combined[combined["name"].isna()]

,id,patient_number,name,gender,diagnosis,surgery,admission_date,discharge_date,total_bill_mk,copay_mk,endoscopy_ledger_balance_mk,invoice_number,source_file,source_sheet
311,15,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,0,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,February
350,19,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,0,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,April
351,20,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,0,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,April
368,17,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,0,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,May
369,18,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,0,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,May
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,24,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
549,25,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
550,26,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December
551,27,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,Endoscopy Ledger_2025 02.04.2026.xlsx,December


In [29]:
endos = []
for endo_file in endo_files:
    endo = pd.read_excel(f"{endo_path}/{endo_files[0]}", engine="openpyxl")
endo = pd.read_excel(f"{endo_path}/{endo_files[0]}", engine="openpyxl")

In [30]:
endo

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,ENDOSCOPY PATIENTS FOR JANUARY 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#,PATIENT NUMBER,NAME,GENDER,DIAGNOSIS,SURGERY,ADMISSION DATE,DISCHARGE DATE,TOTAL BILL (MK),CO-PAY (MK),ENDOSCOPY LEDGER BALANCE (MK),INVOICE NUMBER
3,1,397352,Eliza Evan,Female,Chronic Constipation,Colonoscopy,2025-01-03 00:00:00,2025-01-11 00:00:00,197350,0,197350,398437
4,2,398191,Alinesi Fekinale,Female,Oesophageal Candida,EGD,2025-01-09 00:00:00,2025-01-11 00:00:00,144440,20000,124440,398435
5,3,398805,Annie Willy,Female,Upper and Lower GI Bleed,EGD,2025-01-16 00:00:00,2025-01-17 00:00:00,167700,10000,157700,398509
6,4,398806,Bornface Alick,Male,Dyspepsia,EGD,2025-01-16 00:00:00,2025-01-17 00:00:00,140000,15000,125000,398806
7,5,398697,Wiseman Kambalikana,Male,Anaemia 2 deg UGIB; Acute liver failure,EGD,2025-01-10 00:00:00,2025-01-12 00:00:00,249500,100000,149500,398470
8,6,359515,Wilson Mologeni,Male,Upper GI Bleed,EGD + Banding,2025-01-08 00:00:00,2025-01-10 00:00:00,257100,150000,107100,398398
9,7,387496,Alex Chambaro,Male,UGIB and Previous anemia,Colonoscopy,2025-01-08 00:00:00,2025-01-10 00:00:00,161600,130000,31600,398375


In [34]:
import pandas as pd
import re
from pathlib import Path


def clean_endo_file(filepath: str | Path) -> pd.DataFrame:
    """
    Reads a raw endo Excel/CSV file, finds the header row containing
    '#', 'PATIENT NUMBER', 'NAME', etc., renames columns to snake_case,
    drops rows above the header, and drops rows without an id (#).

    Works for .xlsx, .xls, and .csv files.
    """
    path = Path(filepath)

    # ── 1. Load raw file ──────────────────────────────────────────────
    if path.suffix in (".xlsx", ".xls"):
        raw = pd.read_excel(path, header=None, engine="openpyxl")
    elif path.suffix == ".csv":
        raw = pd.read_csv(path, header=None)
    else:
        # No extension or unknown — try openpyxl first, then CSV
        try:
            raw = pd.read_excel(path, header=None, engine="openpyxl")
        except Exception:
            try:
                raw = pd.read_csv(path, header=None)
            except Exception:
                raise ValueError(
                    f"Could not read '{path.name}' as Excel or CSV. "
                    "Check the file format."
                )

    # ── 2. Find the header row ────────────────────────────────────────
    # The header row is the first row whose first non-null value is '#'
    header_row_idx = None
    for i, row in raw.iterrows():
        values = row.dropna().tolist()
        if values and str(values[0]).strip() == "#":
            header_row_idx = i
            break

    if header_row_idx is None:
        raise ValueError("Could not find the header row (row starting with '#').")

    # ── 3. Rebuild DataFrame from the header row onward ───────────────
    headers = raw.iloc[header_row_idx].tolist()
    data = raw.iloc[header_row_idx + 1 :].copy()
    data.columns = headers
    data = data.reset_index(drop=True)

    # ── 4. Rename columns to snake_case ───────────────────────────────
    def to_snake_case(name: str) -> str:
        name = str(name).strip()
        name = re.sub(r"[^\w\s]", "", name)          # remove punctuation
        name = re.sub(r"\s+", "_", name)              # spaces → underscore
        name = name.lower()
        return name

    data.columns = [to_snake_case(c) for c in data.columns]

    # The '#' column becomes 'id' for clarity
    data = data.rename(columns={"#": "id"})

    # ── 5. Drop rows without a valid id ───────────────────────────────
    data = data[pd.to_numeric(data["id"], errors="coerce").notna()]
    data["id"] = data["id"].astype(int)
    data = data.reset_index(drop=True)

    return data


# ── Batch-process all endo files in a folder ─────────────────────────
def process_all_endo_files(folder: str | Path) -> dict[str, pd.DataFrame]:
    """
    Finds all Excel/CSV files in `folder` whose name contains 'endo'
    (case-insensitive), cleans each one, and returns a dict of
    {filename: cleaned_dataframe}.
    """
    folder = Path(folder)
    results = {}

    for f in sorted(folder.iterdir()):
        if "endo" in f.name.lower() and f.suffix in (".xlsx", ".xls", ".csv"):
            print(f"Processing: {f.name}")
            try:
                df = clean_endo_file(f)
                results[f.name] = df
                print(f"  → {len(df)} rows, columns: {list(df.columns)}")
            except Exception as e:
                print(f"  ✗ Failed: {e}")

    return results



In [35]:


endo_path = "../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger"
endo_files = os.listdir(endo_path)

endos = []
for endo_file in endo_files:
    df = clean_endo_file(f"{endo_path}/{endo_file}")
    endos.append(df)

combined = pd.concat(endos, ignore_index=True)
combined.head()

KeyError: 'id'